In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from vectorization.vectorize import GensimDoc2VecVectorizer, SBERTVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_20newsgroups
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold


C:\Users\fidel\miniconda3\envs\ML-XAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# import datasets

In [2]:
df_sumy = pd.read_csv("../01_Preprocessing/data/20newsgroup_sumy_summary.csv")
df_fulltext = pd.read_csv("../01_Preprocessing/data/20newsgroup_full_text.csv")
df_llm = pd.read_csv("../01_Preprocessing/data/20newsgroup_llm_summary.csv")
df_fulltext["text"] = df_fulltext["text"].fillna("")
df_sumy["summary_66"] = df_sumy["summary_66"].fillna("")
df_llm["summary"] = df_llm["summary"].fillna("")




# Vectorizers

In [3]:
vectorizers = {
        'tfidf': TfidfVectorizer(
            max_features=10000,
            ngram_range=(1,2),
            stop_words='english',
            min_df=2,
            max_df=0.95
        ),
        'sbert': SBERTVectorizer(model_name='all-MiniLM-L6-v2'),
        'doc2vec': GensimDoc2VecVectorizer(vector_size=100, epochs=20)
    }

# CLF

In [4]:
models = {
    'svm': SVC(kernel='linear', random_state=42, probability=True),
    'mlp': MLPClassifier(
        hidden_layer_sizes=(100, 50),
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1
    ),
    'dt': RandomForestClassifier(
        n_estimators=200,
        max_depth=30,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )
}

In [5]:
X = df_sumy["summary_66"]
Y = df_sumy["target"]
df_sumy.count()


target         8794
target_name    8794
full_text      8794
summary_66     8794
summary_33     8794
dtype: int64

In [6]:
import os
import pickle

output_dir = "saved_pipelines_20news/sumy"
for vect_name, vect in vectorizers.items():
    for model_name, model in models.items():
        print(f"Training: {vect_name} + {model_name}")

        pipeline = Pipeline([
            ('vectorizer', vect),
            ('classifier', model)
        ])

        try:
            pipeline.fit(X, Y)

            filename = f"{vect_name}_{model_name}_sumy.pkl"
            filepath = os.path.join(output_dir, filename)

            with open(filepath, 'wb') as f:
                pickle.dump(pipeline, f)

            print(f"Saved: {filename}")
        except Exception as e:
            print(f"Failed: {vect_name} + {model_name} — {e}")

Training: tfidf + svm
Saved: tfidf_svm_sumy.pkl
Training: tfidf + mlp
Saved: tfidf_mlp_sumy.pkl
Training: tfidf + dt
Saved: tfidf_dt_sumy.pkl
Training: sbert + svm


Batches: 100%|██████████| 275/275 [02:55<00:00,  1.57it/s]


Saved: sbert_svm_sumy.pkl
Training: sbert + mlp


Batches: 100%|██████████| 275/275 [03:07<00:00,  1.47it/s]


Saved: sbert_mlp_sumy.pkl
Training: sbert + dt


Batches: 100%|██████████| 275/275 [03:08<00:00,  1.46it/s]


Saved: sbert_dt_sumy.pkl
Training: doc2vec + svm
Saved: doc2vec_svm_sumy.pkl
Training: doc2vec + mlp
Saved: doc2vec_mlp_sumy.pkl
Training: doc2vec + dt
Saved: doc2vec_dt_sumy.pkl
